# Maintainers

Release wheels can include a Linux x86_64 executable built from the
pinned upstream C model. The public user workflow is Python-first;
this notebook explains the maintainer machinery that makes the
bundled executable auditable.


In [ ]:
from ebolasim.build import detect_platform_id, read_patch_inventory, read_source_lock

lock = read_source_lock()
patches = read_patch_inventory()
{
    "platform_id": detect_platform_id(),
    "lock_placeholder": lock.is_placeholder,
    "repository": lock.repository,
    "ref_type": lock.ref_type,
    "ref": lock.ref,
    "patches": [patch["file"] for patch in patches.to_dict()["patches"]],
}


## The CI Pipeline In Package Functions

The release pipeline repeats these steps:

1. `read_source_lock()` reads `model-src/upstream.lock.yml`.
2. `fetch_source()` downloads the pinned archive and verifies SHA256.
3. `apply_patches()` applies the patch inventory in order.
4. `build_executable()` compiles `ebola-spatial-linux`.
5. `ebolasim.Sim(...).run()` performs a small smoke simulation.
6. `bundle_executable()` stages the binary inside `src/ebolasim/_bundled/linux-x86_64/`.
7. `tools/ci/build_release_bundle.py` writes metadata, checksums,
   and a separate GitHub release bundle.

Run the same path locally with uv:


In [ ]:
commands = [
    "uv sync --extra dev --extra docs --extra plot",
    "uv run ebolasim source show --pretty",
    "uv run ebolasim source fetch --out build/upstream --pretty",
    "uv run ebolasim source build "
    "build/upstream/extract/ebolasim_public-... "
    "--out build/linux --overwrite",
    "uv run python tools/ci/build_release_bundle.py "
    "--overwrite --package-root src/ebolasim",
    "uv build",
]
print("\n".join(commands))


## Updating The Source Lock

Keep both lock files synchronized:

- `model-src/upstream.lock.yml`
- `src/ebolasim/_patches/upstream.lock.yml`

Record the upstream repository URL, exact commit or tag, archive
URL, SHA256 command output, reason for selecting the ref, CI bundle
artifact review, and a fresh wheel install smoke run.

A release tag must be `vX.Y.Z` and match both `pyproject.toml` and
`ebolasim.__version__`.
